# Problem 7 (100 points)

Parameter counting is one of the most fundamental skills in deep learning competitions. Given an architecture specification, you must be able to compute the exact number of learnable parameters, trace tensor shapes through every layer, and estimate computational cost. In this problem, you will derive parameter counts analytically, implement a general-purpose parameter counter, and trace shapes through complex architectures.

We use the following notation in this problem.
- `Linear(in, out)`: $\text{in} \times \text{out} + \text{out}$ parameters (with bias).
- `Conv2d(C_in, C_out, K)`: $C_{\text{out}} \times C_{\text{in}} \times K^2 + C_{\text{out}}$ parameters (with bias).
- `Conv2d(C_in, C_out, K, groups=g)`: $C_{\text{out}} \times (C_{\text{in}} / g) \times K^2 + C_{\text{out}}$ parameters.
- `BatchNorm(C)`: $2C$ learnable parameters ($\gamma$ and $\beta$).
- ReLU, MaxPool, Dropout, Flatten: **0** learnable parameters.

In [ ]:
# Run code in this cell

"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""

import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(42)

> WARNING !!!
>
- Beyond importing libraries/modules/classes/functions in the preceding cell, you are **NOT allowed to import anything else for the following purposes**:
    - **As a part of your final solution.**
    - **Temporarily import something to assist you to get a solution.**

## Part 1 (10 points, coding task)

**Do the following tasks.**

Implement helper functions that compute parameter counts **analytically** (no model instantiation needed).

1. `linear_params(in_f, out_f, bias=True)` — for `nn.Linear`.
2. `conv2d_params(c_in, c_out, k, bias=True, groups=1)` — for `nn.Conv2d`.
3. `batchnorm_params(c)` — learnable parameter count for `nn.BatchNorm{1d,2d}`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def linear_params(in_f, out_f, bias=True):
    ...

def conv2d_params(c_in, c_out, k, bias=True, groups=1):
    ...

def batchnorm_params(c):
    ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
assert linear_params(784, 256) == sum(p.numel() for p in nn.Linear(784, 256).parameters())
assert linear_params(256, 10, bias=False) == sum(p.numel() for p in nn.Linear(256, 10, bias=False).parameters())
assert conv2d_params(3, 64, 3) == sum(p.numel() for p in nn.Conv2d(3, 64, 3).parameters())
assert conv2d_params(64, 64, 3, bias=False) == sum(p.numel() for p in nn.Conv2d(64, 64, 3, bias=False).parameters())
assert conv2d_params(32, 32, 3, groups=32) == sum(p.numel() for p in nn.Conv2d(32, 32, 3, groups=32).parameters())
assert batchnorm_params(64) == sum(p.numel() for p in nn.BatchNorm2d(64).parameters())
print("Part 1 passed!")

Using these helper functions, let us count parameters for three concrete architectures.

## Part 2 (20 points, non-coding task)

**Do the following tasks (Reasoning is required).**

Compute the **exact** total parameter count for each architecture below. Show your work.

**Architecture A** — Simple MLP:
```
Linear(784, 512) -> ReLU -> Linear(512, 256) -> ReLU -> Linear(256, 10)
```

**Architecture B** — Simple CNN:
```
Conv2d(3, 32, 3, pad=1) -> BN(32) -> ReLU -> MaxPool(2) ->
Conv2d(32, 64, 3, pad=1) -> BN(64) -> ReLU -> MaxPool(2) ->
Conv2d(64, 128, 3, pad=1) -> BN(128) -> ReLU -> AdaptiveAvgPool(1) ->
Linear(128, 10)
```

**Architecture C** — ResNet bottleneck block:
```
Conv2d(256, 64, 1, bias=False) -> BN(64) ->
Conv2d(64, 64, 3, pad=1, bias=False) -> BN(64) ->
Conv2d(64, 256, 1, bias=False) -> BN(256)
```

Store your answers as `params_A`, `params_B`, `params_C` (integers).

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

In [ ]:
### WRITE YOUR SOLUTION HERE ###

params_A = ...  # int
params_B = ...  # int
params_C = ...  # int

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
model_A = nn.Sequential(
    nn.Linear(784, 512), nn.ReLU(), nn.Linear(512, 256), nn.ReLU(), nn.Linear(256, 10))
actual_A = sum(p.numel() for p in model_A.parameters())
assert params_A == actual_A, f"A: expected {actual_A}, got {params_A}"

model_B = nn.Sequential(
    nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
    nn.Flatten(), nn.Linear(128, 10))
actual_B = sum(p.numel() for p in model_B.parameters())
assert params_B == actual_B, f"B: expected {actual_B}, got {params_B}"

model_C = nn.Sequential(
    nn.Conv2d(256, 64, 1, bias=False), nn.BatchNorm2d(64),
    nn.Conv2d(64, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64),
    nn.Conv2d(64, 256, 1, bias=False), nn.BatchNorm2d(256))
actual_C = sum(p.numel() for p in model_C.parameters())
assert params_C == actual_C, f"C: expected {actual_C}, got {params_C}"

print(f"Part 2 passed! A={params_A:,}, B={params_B:,}, C={params_C:,}")

Rather than counting by hand every time, let us build a tool that counts parameters for any model.

## Part 3 (15 points, coding task)

**Do the following tasks.**

Implement `count_model_params(model)` that traverses any `nn.Module` and returns a breakdown.

Return a dict with:
- `'total'`: total parameter count (int).
- `'trainable'`: count of parameters with `requires_grad=True` (int).
- `'non_trainable'`: count of parameters with `requires_grad=False` (int).
- `'per_layer'`: list of `(name, shape_tuple, count, is_trainable)` tuples.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def count_model_params(model):
    """
    Count parameters of any nn.Module.
    Returns: dict with 'total', 'trainable', 'non_trainable', 'per_layer'
    """
    ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
import torchvision
resnet = torchvision.models.resnet18(weights=None)

# Freeze some layers
for name, param in resnet.named_parameters():
    if 'layer1' in name or 'conv1' in name or 'bn1' in name:
        param.requires_grad = False

info = count_model_params(resnet)
assert info['total'] == sum(p.numel() for p in resnet.parameters())
assert info['trainable'] == sum(p.numel() for p in resnet.parameters() if p.requires_grad)
assert info['non_trainable'] == sum(p.numel() for p in resnet.parameters() if not p.requires_grad)
assert info['total'] == info['trainable'] + info['non_trainable']
assert len(info['per_layer']) > 0
print(f"Part 3 passed! ResNet-18: {info['total']:,} total, {info['trainable']:,} trainable")

Beyond counting parameters, tracing shapes is equally important for debugging architectures.

## Part 4 (15 points, coding task)

**Do the following tasks.**

Implement `trace_shapes(model, input_shape)` that passes a dummy tensor through the model and records the output shape after each leaf module.

- `model`: any `nn.Module`.
- `input_shape`: tuple, e.g., `(1, 3, 32, 32)`.
- Returns: list of `(layer_name, output_shape_tuple)` pairs.

Hint: use `register_forward_hook` on every leaf module.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def trace_shapes(model, input_shape):
    """
    Trace output shapes through a model.
    Returns: list of (layer_name, output_shape_tuple)
    """
    ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
test_model = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1),
    nn.ReLU(),
    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(),
    nn.Linear(32, 10),
)

shapes = trace_shapes(test_model, (1, 3, 32, 32))
assert len(shapes) >= 4, f"Expected at least 4 shape entries, got {len(shapes)}"
assert shapes[-1][1][-1] == 10, "Final output should have 10 classes"
print("Part 4 passed!")
for name, shape in shapes:
    print(f"  {name}: {shape}")

Finally, let us tackle conceptual questions about parameter efficiency in modern architectures.

## Part 5 (10 points, non-coding task)

**Do the following tasks (Reasoning is required).**

1. VGG-16 has ~138M parameters, of which ~124M are in the three fully connected layers. ResNet-50 has ~25.6M parameters with no large FC layers (it uses Global Average Pooling). Explain how ResNet achieves better accuracy with 5x fewer parameters.

2. A **depthwise separable convolution** (MobileNet) replaces `Conv2d(C_in, C_out, K)` with two layers: `Conv2d(C_in, C_in, K, groups=C_in)` followed by `Conv2d(C_in, C_out, 1)`. Compute the parameter ratio $\frac{\text{standard}}{\text{depthwise separable}}$ for $C_{\text{in}} = 64$, $C_{\text{out}} = 128$, $K = 3$ (all with bias). Show your calculation.

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 6 (15 points, coding task)

**Do the following tasks.**

Compute the total number of **multiply-accumulate operations (MACs)** for a single forward pass through a `Conv2d(C_in, C_out, K, padding=P, stride=S)` layer on input shape `(1, C_in, H, W)`.

The formula is:

$$\text{MACs} = C_{\text{out}} \times H_{\text{out}} \times W_{\text{out}} \times C_{\text{in}} \times K^2$$

Implement `conv2d_macs(c_in, c_out, k, h_in, w_in, padding=0, stride=1)` and use it to compute MACs for the first convolutional layer of ResNet-18: `Conv2d(3, 64, 7, stride=2, padding=3)` on a $224 \times 224$ input. Store the result as `resnet_conv1_macs`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def conv2d_macs(c_in, c_out, k, h_in, w_in, padding=0, stride=1):
    ...

resnet_conv1_macs = ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
# Simple test: Conv2d(1, 1, 3) on 5x5 input, no padding, stride 1
# Output: 3x3, MACs = 1 * 3 * 3 * 1 * 9 = 81
assert conv2d_macs(1, 1, 3, 5, 5) == 81, f"Expected 81, got {conv2d_macs(1, 1, 3, 5, 5)}"

# ResNet conv1: Conv2d(3, 64, 7, stride=2, padding=3) on 224x224
# Output: 112x112, MACs = 64 * 112 * 112 * 3 * 49 = 118,013,952
expected_macs = 64 * 112 * 112 * 3 * 49
assert resnet_conv1_macs == expected_macs, f"Expected {expected_macs:,}, got {resnet_conv1_macs:,}"
print(f"Part 6 passed! ResNet conv1 MACs: {resnet_conv1_macs:,}")

## Part 7 (15 points, non-coding task)

**Do the following tasks (Reasoning is required).**

In a transfer learning scenario, you load a pretrained ResNet-18 (11,689,512 total parameters) and replace the final `fc` layer (`Linear(512, 1000)`) with `Linear(512, 5)` for a 5-class task, freezing all other layers.

1. How many parameters does the **original** `fc` layer have (with bias)?
2. How many parameters does the **new** `fc` layer have (with bias)?
3. How many total parameters does the modified model have?
4. How many parameters are **trainable** (not frozen)?
5. What percentage of total parameters are trainable? Is this efficient?

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """